In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import time
import tqdm
import helpers

import environment
import environment.wrappers
import environment.bicycle as bicycle

import controllers.dqn as dqn
import controllers.ppo as ppo
import controllers.clothoids as clothoids
import controllers.purepursuit as purepursuit

import metrics

import numpy as np
import gymnasium as gym
import matplotlib.pyplot as plt

In [ ]:
env = environment.bicycle.BicycleCarEnv(
    road_network=bicycle.RoadNetwork(roads=[
        bicycle.create_rectangular_track(
            center=(50.0, 50.0),
            length=80.0,
            height=40.0,
            turn_radius=8.0,
            width=8.0,
        )
    ]),
    render_mode="rgb_array",
    spawn=((50.0, 30.0), 0.0),
    goal=((10.0, 50.0), 2.0),
    obstacles=[
        bicycle.Circle(center=(90, 50), radius=1.0),
    ]
)

env = environment.wrappers.observations.WithBaseInfo(env)
env = environment.wrappers.observations.WithObstaclesInfo(
    env,
    detection_range=50.0,
    max_obstacles=10,
)
env = environment.wrappers.observations.WithPathInfo(
    env,
    num_waypoints=100
)
env = environment.wrappers.observations.WithRoadInfo(
    env,
    num_boundary_points=20,  # Increased for better boundary coverage
    lookahead_distance=40.0,  # Increased lookahead for tentacles
)
env = environment.wrappers.observations.WithDynamicsInfo(
    env,
    include_history=3,
)

controller = clothoids.ClothoidTentaclesController(
    env=env,
    num_tentacles=41,
    t0=10.0,
    l0=10.0,
    min_tentacle_length=10.0,  # Ensure reasonable tentacle length at low speeds
    num_points_per_tentacle=64,

    # clearance, curvature, trajectory
    weights=(0.1, 0.2, 0.5),

    target_velocity=3.5,
)


print("[env.observation_space]:", env.observation_space)
print("[env.action_space]:", env.action_space)

print("[path.#points]:", len(env.unwrapped.path))

observation, info = env.reset()

print("[observation]:", observation.keys())

helpers.preview(env, False)

In [ ]:
# controller = purepursuit.PurePursuitController()

observation, info = env.reset()

for i in range(500):
    # TODO: what are the _states ?
    action, _states = controller.get_action(observation=observation)
    
    print(action)
    
    observation, reward, terminated, truncated, info = env.step(action)
    
    env.unwrapped.overlay_manager.clear()
    controller.draw_debug(env, observation=observation, path=env.unwrapped.path)
    
    helpers.preview(env)

    if terminated or truncated:
        break

env.close()

In [ ]:
episode_data = env.unwrapped.recorder.to_arrays()

cte_metrics = metrics.compute_cross_track_error(
    positions=episode_data['positions'],
    reference_path=env.unwrapped.path,
)
print("[cte-rms]:", f"{cte_metrics['cte_rms']:.3f}", "m")

smoothness_metrics = metrics.compute_steering_smoothness(
    steering_angles=episode_data['steering_angles'],
    dt=env.unwrapped.DELTA_TIME
)
print(f"Steering Jerk RMS: {smoothness_metrics['steering_jerk_rms']:.3f} rad/s³")

success = metrics.compute_success_rate([episode_data])
print(f"Success: {'Yes' if success == 1.0 else 'No'}")